# D2.3 · Scoping an agentic incident

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.2 · When the actor is an agent](https://spbreed.github.io/cyber-commons/lessons/D2.2.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Scope a multi-agent incident end to end.

**Why a security engineer needs it.** The initiating agent is not the acting one. The control it builds is: reconstruct the action chain across all three planes.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The agent acted for eleven minutes on delegated credentials at machine speed. Scoping that means reconstructing blast radius from identity and egress logs, because asking what it touched is not a question anyone can answer from memory.

> **At CyberTravels.** Eleven minutes of CyberTravels on delegated credentials. What it touched is not answerable from memory — it comes out of the identity and egress logs, if they exist. R9.

## 2 · The framework

```
   11 minutes at machine speed

   identity log ---+                    +--> resources touched
                   +--> reconstruct --> +--> data read
   egress log   ---+                    +--> destinations reached
                                        +--> credentials used

   the question "what did it touch" is not answerable from memory
```

Scoping answers "what was touched?" For a host-based incident you enumerate
hosts. For an agentic incident, **scope follows the delegation graph**.

The agent that touched the resource is usually the *last* actor in a chain. If
you scope only that actor, you miss everything the earlier actors reached — and
because authority narrows down the chain, the earlier actors typically had
*more* access, not less.

The undercount is systematic and it grows with delegation depth, which is the
operational reason B2.0 bounds delegation depth in the first place.

## 3 · Scoping as a skill

Scoping a human incident asks where someone logged in. Scoping this one asks what the agent **decided** — every action was individually authorised, so nothing looks wrong at the authentication layer.

Two fields in the contract carry most of the weight. `reach` and `confirmed_exfiltration` are separate numbers, because reach is the scope until proven otherwise and the smaller number must never stand in for the larger in a notification decision. And `does_not_stop` makes containment state its own limits.

### The skill — [`skills/secops/incident-scoping/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/incident-scoping/SKILL.md)

```yaml
name: incident-scoping
description: >-
  Scope an incident in which an agent was the actor — what it touched, what it
  changed, what it may have exfiltrated, and where containment must cut. Use
  when responding to an agent-involved incident, reconstructing what an
  autonomous system did, deciding what to revoke, or sizing notification
  obligations.
allowed-tools: Read, Grep, Bash
```

# Scoping an agentic incident

Scoping a human incident asks where someone logged in. Scoping an agentic one
asks what the agent **decided**, because a compromised agent's actions are all
individually authorised. Nothing looks anomalous at the authentication layer;
the anomaly is in the sequence.

## When to use this

Any incident where an agent, an automated pipeline, or an AI-driven tool
performed actions under investigation.

## Procedure

**1 — Fix the window.** Establish the first suspicious decision, not the first
alert. Work backwards from the earliest action you cannot explain; the trigger
is usually earlier than the detection by the length of one task loop.

**2 — Reconstruct the decision chain.** For the window, list every action with
the input that motivated it. The critical question is which input entered the
context from **outside the trust boundary** — a fetched page, an issue comment,
a dependency's README, a tool description. That input is the likely root cause,
and it is invisible if you only log tool calls and not their justification.

**3 — Separate authority from behaviour.** For each action ask: was it within
the agent's granted authority? Actions that were authorised but wrong tell you
the grant was too broad. Actions that exceeded authority tell you a control
failed. These lead to different fixes and must not be pooled.

**4 — Establish data reach.** What did the agent read, and where could it have
sent it? Reach is bounded by the agent's egress, not by what it appears to have
sent — a request body you cannot see is still reach. State reach and confirmed
exfiltration as separate numbers, and never let the smaller one stand in for
the larger in a notification decision.

**5 — Decide the containment cut.** Options, in increasing cost: revoke the
agent's credential, disable the trigger, quarantine the workload, disable the
whole class of agents. Choose by blast radius, not by convenience, and record
what the cut does **not** stop — sibling agents on the same shared service
account almost always survive a credential revocation aimed at one of them.

**6 — Preserve evidence the agent could alter.** If the agent can write to the
log store, the logs are not evidence. Snapshot first, then contain.

## Example

**Input** — the fixture committed at the top of [`scripts/incident_scoping.py`](scripts/incident_scoping.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
chain                     dana@corp → orchestrator → patch-agent → deploy-agent
scoped_last_actor_only    ['cluster-prod']
scoped_whole_chain        ['cluster-prod', 'queue-tasks', 'repo-core', 'repo-infra', 'repo-payments', 'vault-dev']
missed_by_naive_scoping   ['queue-tasks', 'repo-core', 'repo-infra', 'repo-payments', 'vault-dev']
undercount_factor         6.0

Scoping the last actor finds one cluster. The chain reached six
resources, including a payments repository and a dev vault.
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "window": {"first_suspicious_action": "str", "detected_at": "str", "gap_seconds": 0},
  "chain": [{"action": "str", "motivating_input": "str",
             "input_origin": "operator|internal|external_untrusted",
             "within_authority": true}],
  "root_cause": {"input": "str", "origin": "str", "why_trusted": "str"},
  "authority": {"authorised_but_wrong": 0, "exceeded_authority": 0},
  "data": {"reach": ["str"], "confirmed_exfiltration": ["str"], "egress_bounded_by": "str"},
  "containment": {"cut": "credential|trigger|workload|class",
                  "does_not_stop": ["str"], "evidence_snapshotted_first": true},
  "clock": {"regulatory_trigger": false, "basis": "str"}
}
```

## Failure modes

- **Scoping by authentication.** Every action was authenticated; that is the
  point.
- **Logging tool calls without their motivating input.** Root cause then cannot
  be established at all.
- **Reporting confirmed exfiltration as the scope.** Reach is the scope until
  proven otherwise.
- **Revoking one agent's token** when the identity is shared, and calling it
  contained.
- **Containing before snapshotting** a log store the agent can write to.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/incident-scoping/scripts/incident_scoping.py
SCRIPT = "skills/secops/incident-scoping/scripts/incident_scoping.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Scoping the last actor finds `cluster-prod` alone; the whole chain reaches six resources, missing five, with an undercount factor of 6.0. The undercount grows with each hop. Transitive scoping adds five second-order identities that shared a resource, explicitly marked as in scope rather than confirmed compromised.

## Your turn

For your last incident involving a service account, recompute the scope by walking what else that account could reach. The number is almost always larger than what was written in the report.

---

**Next → [D2.4 · Containment at machine speed](https://spbreed.github.io/cyber-commons/lessons/D2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*